# Lexical Diversity and Sentiment as Low-Dimensional Text Features

A common baseline for text classification is TF-IDF: a high-dimensional bag-of-words
representation that weighs each term by its frequency and rarity across documents. This
notebook tests an alternative, much lower-dimensional representation for a binary text
classification task, combining lexical diversity (MATTR) with sentiment (VADER), 
and reports honestly on where a 5-feature model can and can't compete with a
high-dimensional one.

The task: classify blog posts into one of two groups, using only 5 engineered features
per document instead of thousands of TF-IDF terms.


In [ ]:
import re
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lexical_diversity import lex_div as ld
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 1. Data

Each document in the source corpus is wrapped in `<doc date="...">...</doc>` tags, one
file per class. The parser below reads a file and returns one record per document with
its date, raw text, and class label.

Raw corpus files are not included in this repository (see `data/README.md` for
details and access notes). Place them under `data/raw/` before running this notebook.


In [ ]:
def parse_corpus(filepath, label):
    """Parses a corpus file with <doc date="...">...</doc>-delimited posts.

    Returns a list of dicts: {date, text, label}.
    """
    with open(filepath, encoding="utf-8") as f:
        content = f.read()

    pattern = re.compile(r'<doc date="([^"]*)">(.*?)</doc>', re.DOTALL)
    records = []
    for date, text in pattern.findall(content):
        records.append({"date": date, "text": text.strip(), "label": label})
    return records


records = parse_corpus("data/raw/class1.txt", "class1")
records += parse_corpus("data/raw/class2.txt", "class2")

print(f"Parsed {len(records)} documents")
print(f"class1: {sum(1 for r in records if r['label'] == 'class1')}")
print(f"class2: {sum(1 for r in records if r['label'] == 'class2')}")


## 2. Feature extraction: MATTR + VADER

Two feature families, five numbers per document total:

- **MATTR** (Moving-Average Type-Token Ratio): a length-independent measure of
  vocabulary diversity. Computed on lowercased, punctuation-stripped text since
  lexical diversity should not depend on casing or punctuation.
- **VADER**: four sentiment scores (negative, neutral, positive, compound). Computed
  on the *original* text, since punctuation and capitalization carry sentiment
  information that the MATTR preprocessing deliberately strips out.


In [ ]:
def tokenize_for_mattr(text):
    """Lowercase, remove punctuation, split on whitespace."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()


def extract_features(records, mattr_window=50):
    analyzer = SentimentIntensityAnalyzer()
    rows = []

    for r in records:
        tokens = tokenize_for_mattr(r["text"])
        if len(tokens) < 2:
            continue  # too short to compute a meaningful MATTR score

        mattr_score = ld.mattr(tokens, window_length=mattr_window)
        vader_scores = analyzer.polarity_scores(r["text"])

        rows.append({
            "mattr": mattr_score,
            "vader_neg": vader_scores["neg"],
            "vader_neu": vader_scores["neu"],
            "vader_pos": vader_scores["pos"],
            "vader_compound": vader_scores["compound"],
            "label": r["label"],
        })

    return pd.DataFrame(rows)


features_df = extract_features(records)
features_df.describe()


In [ ]:
X = features_df.drop(columns=["label"])
y = features_df["label"]


## 3. Model: logistic regression with cross-validated regularization

Logistic regression is a sensible choice here given how few features we have. A grid
search over `C` (inverse regularization strength) picks the value that maximizes
F1-macro under 5-fold cross-validation. `class_weight="balanced"` compensates for the
class imbalance in the corpus, and features are standardized within each fold to avoid
leaking test-set statistics into the scaler.


In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
cv_results = []

for C in C_values:
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            C=C,
            penalty="l2",
            solver="liblinear",
            max_iter=2000,
            class_weight="balanced",
        ))
    ])
    scores = cross_val_score(clf, X, y, cv=5, scoring="f1_macro")
    cv_results.append((C, scores.mean(), scores.std()))
    print(f"C={C:<7} -> F1-macro: {scores.mean():.3f} +/- {scores.std():.3f}")

best_C, best_f1, best_std = max(cv_results, key=lambda r: r[1])
print(f"\nBest C = {best_C} | F1-macro = {best_f1:.3f} +/- {best_std:.3f}")


In [ ]:
results_df = pd.DataFrame(cv_results, columns=["C", "f1_macro_mean", "f1_macro_std"])

plt.figure(figsize=(7, 5))
plt.errorbar(results_df["C"], results_df["f1_macro_mean"], yerr=results_df["f1_macro_std"],
             marker="o", linestyle="--", capsize=4)
plt.xscale("log")
plt.xlabel("C (regularization strength, log scale)")
plt.ylabel("F1-macro")
plt.title("F1-macro vs. regularization parameter C")
plt.tight_layout()
plt.show()


## 4. Evaluation at the best C

We refit the pipeline at the best `C` and look at out-of-fold predictions
(`cross_val_predict`) for a confusion matrix and per-class metrics.


In [ ]:
best_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        C=best_C,
        penalty="l2",
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced",
    ))
])

y_pred = cross_val_predict(best_clf, X, y, cv=5)

cm = confusion_matrix(y, y_pred, labels=["class1", "class2"])
print("Confusion matrix (rows = true, columns = predicted):")
print(cm)

print("\nClassification report:")
print(classification_report(y, y_pred, target_names=["class1", "class2"]))


In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["class1", "class2"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix (C = {best_C})")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200)
plt.show()


## Notes on interpreting the results

If `C` barely moves the F1-macro score across several orders of magnitude, that's a
signal worth taking seriously: it points to a ceiling in the features themselves
rather than a regularization problem. In that case, per-class precision and recall
close to each other (rather than one much higher than the other) suggest the model
isn't finding a genuine signal, just trading recall on one class for precision on
the other. See `docs/report.md` for the full discussion, including concrete next
steps (adding TF-IDF or transformer-based embeddings, controlling for author overlap
between train and test folds with grouped cross-validation, and testing non-linear
classifiers once more features are added).
